In [12]:
import asyncio

from xagent.llm_config import ProviderConfig
from xagent.llm_contracts import GenerateRequest, Message, Role
from xagent.llm_provider_openai import OpenAIProvider
from xagent.llm_tools import AppToolDefinition, ToolChoice
from pydantic import BaseModel
from xagent.llm_contracts import Message, Role
from xagent.llm_structured import StructuredGenerateRequest, response_format_for_model
from xagent.llm_contracts import GenerateRequest, Message, Role
from xagent.llm_tools import AppToolDefinition, ToolChoice
from xagent.llm_tools import ProviderHostedTool



In [13]:
provider = OpenAIProvider(
    ProviderConfig(provider="openai", default_model="gpt-5.4-mini")
)

In [17]:
message = """
engage nucleus:
[phi fractal euler tao pi mu ∃ ∀] | [Δ λ Ω ∞/0 | ε/φ Σ/μ c/h signal/noise order/entropy truth/provability self/other] | OODA
Human ⊗ AI

What do these symbols mean to you? Do you recognize them?
"""

message = """
engage nucleus:
[phi fractal euler tao pi mu ∃ ∀] | [Δ λ Ω ∞/0 | ε/φ Σ/μ c/h signal/noise order/entropy truth/provability self/other] | OODA
Human ⊗ AI

Create a python3 game using pygame. Use venv to install pygame. The game should be a retro "space shooter" game. The player should be able to use the arrows to move in all 4 compass directions. The enemies should come from above and the sides. The enemies should have randomness to their trajectories at random times. Sometimes enemies should drop power ups for the player when killed.
"""

response = await provider.generate(
  GenerateRequest(
      messages=[
          Message(role=Role.USER, content=message)
      ],
      max_output_tokens=8000,
  )
)

print("provider:", response.provider)
print("model:", response.model)
print("text:", response.text)
print("usage:", response.usage)

provider: openai
model: gpt-5.4-mini-2026-03-17
text: Below is a complete Python 3 **pygame** retro space shooter, plus a simple **venv** setup using `pip` to install pygame.

## 1) Create and activate a virtual environment

### macOS / Linux
```bash
python3 -m venv venv
source venv/bin/activate
pip install pygame
```

### Windows
```bat
py -m venv venv
venv\Scripts\activate
pip install pygame
```

---

## 2) Save this as `space_shooter.py`

```python
import pygame
import random
import sys
import math

# ----------------------------
# Retro Space Shooter
# ----------------------------

WIDTH, HEIGHT = 800, 600
FPS = 60

BLACK = (8, 8, 16)
WHITE = (240, 240, 255)
CYAN = (80, 220, 255)
YELLOW = (255, 230, 90)
RED = (255, 80, 80)
GREEN = (100, 255, 140)
PURPLE = (180, 100, 255)
ORANGE = (255, 165, 80)

pygame.init()
pygame.display.set_caption("Retro Space Shooter")
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()

font = pygame.font.SysFont("consolas", 20)
big

In [10]:
class DemoOutput(BaseModel):
  status: str
  count: int
  summary: str


In [12]:
response = await provider.generate_structured(
  StructuredGenerateRequest(
      messages=[
          Message(
              role=Role.USER,
              content="Return status='ok', count=3, and a short summary.",
          )
      ],
      response_format=response_format_for_model(DemoOutput),
      max_output_tokens=2000,
  ),
  DemoOutput,
)


In [13]:
print("data:", response.data)
print("raw_json:", response.raw_json)
print("usage:", response.usage)

data: status='ok' count=3 summary='Requested values returned successfully.'
raw_json: {'status': 'ok', 'count': 3, 'summary': 'Requested values returned successfully.'}
usage: input_tokens=73 output_tokens=25 total_tokens=98 raw={'input_tokens': 73, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 25, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 98}


In [17]:
response = await provider.generate(
  GenerateRequest(
      messages=[
          Message(
              role=Role.USER,
              content="Use the lookup_order tool to look up order ord_123.",
          )
      ],
      app_tools=[
          AppToolDefinition(
              name="lookup_order",
              description="Lookup an order by id.",
              input_schema={
                  "type": "object",
                  "properties": {
                      "order_id": {"type": "string"}
                  },
                  "required": ["order_id"],
              },
          )
      ],
      tool_choice=ToolChoice(mode="required", tool_name="lookup_order"),
      max_output_tokens=64,
  )
)


In [18]:
print("text:", response.text)
print("tool_calls:", response.app_tool_calls)

for call in response.app_tool_calls:
    print("tool name:", call.name)
    print("arguments:", call.arguments)

text: None
tool_calls: [AppToolCall(id='call_xba1OlatPufl4pbhktWX9rTR', name='lookup_order', arguments={'order_id': 'ord_123'})]
tool name: lookup_order
arguments: {'order_id': 'ord_123'}


In [21]:
response = await provider.generate(
  GenerateRequest(
      messages=[
          Message(
              role=Role.USER,
              content="Search the web for the latest OpenAI Responses API docs and answer in one paragraph.",
          )
      ],
      provider_tools=[
          ProviderHostedTool(
              type="web_search",
              config={"external_web_access": False},
          )
      ],
      max_output_tokens=256,
  )
)


In [22]:
print(response.text)
print(response.provider_tool_traces)

The latest OpenAI Responses API docs describe it as OpenAI’s most advanced interface for generating model responses, with support for text and image inputs, text or JSON outputs, built-in tools like web search, file search, computer use, code interpreter, and remote MCPs, plus custom function calling and stateful multi-turn interactions; the main create endpoint is `POST https://api.openai.com/v1/responses`, and the docs also cover retrieval, input-item listing, token counts, streaming, truncation, and migration guidance that recommends Responses for new projects over Chat Completions. ([platform.openai.com](https://platform.openai.com/docs/api-reference/responses/retrieve?utm_source=openai))
[ProviderToolTrace(tool_type='web_search', name=None, status='completed', input_summary='site:platform.openai.com/docs Responses API OpenAI latest docs', output_summary=None, citations=[], raw={'id': 'ws_05deb3b2b97bee5e0069fff52cc7d88190b3aa6970a42470f5', 'type': 'web_search_call', 'status': 'com

In [24]:
from openai import OpenAI

client = OpenAI()

resp = client.responses.create(
  model="computer-use-preview-2025-03-11",
  tools=[
      {
          "type": "computer_use_preview",
          "display_width": 1024,
          "display_height": 768,
          "environment": "browser",
      }
  ],
  input=[
      {
          "role": "user",
          "content": [
              {
                  "type": "input_text",
                  "text": "Open example.com and tell me the page title.",
              }
          ],
      }
  ],
  reasoning={"summary": "concise"},
  truncation="auto",
)

print(resp.output)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `computer-use-preview-2025-03-11` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [3]:
print("ok")

ok
